# Sesión 15 — Regularización, Práctica de Entrenamiento e Interpretabilidad
### Reconocimiento de Patrones y Aprendizaje Automático — Posgrado en Ingeniería Biomédica

**Módulo IV · Fundamentos de Deep Learning — Sesión de cierre**

## Objetivos de aprendizaje

1. Comprender dropout, batch normalization y layer normalization — dónde y por qué usar cada una.
2. Aplicar estrategias de aumentación de datos específicas para señales fisiológicas e imágenes médicas.
3. Implementar calentamiento (warmup) de tasa de aprendizaje y recocido coseno.
4. Implementar **Grad-CAM** desde cero para generar mapas de activación de clase en ECG y fMRI.
5. Comprender cuándo los mapas de saliencia son confiables y cuándo engañan.

## Lecturas recomendadas

| Prioridad | Referencia |
|---|---|
| ★★★ | Srivastava, N. et al. (2014). Dropout: a simple way to prevent neural networks from overfitting. *JMLR*, 15. |
| ★★★ | Selvaraju, R.R. et al. (2017). Grad-CAM: visual explanations from deep networks. *ICCV*. |
| ★★☆ | Ioffe, S. & Szegedy, C. (2015). Batch normalization: accelerating deep network training. *ICML*. |
| ★★☆ | Loshchilov, I. & Hutter, F. (2017). SGDR: stochastic gradient descent with warm restarts. *ICLR*. |
| ★★☆ | Adebayo, J. et al. (2018). Sanity checks for saliency maps. *NeurIPS*. — Lectura crítica sobre la confiabilidad de la saliencia. |
| ★☆☆ | Cubuk, E.D. et al. (2019). AutoAugment: learning augmentation strategies from data. *CVPR*. |

## Parte 0 — Configuración

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import warnings; warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

rng    = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11,
})
print(f'Dispositivo: {device}')

## Parte 1 — Dropout y batch normalization

**Dropout** pone a cero aleatoriamente activaciones con probabilidad $p$ durante el entrenamiento:
- Ensamble efectivo de $2^n$ redes que comparten pesos
- Reduce la co-adaptación de neuronas
- *Siempre desactivar durante evaluación* (`model.eval()`)

**Batch normalization** normaliza las pre-activaciones dentro de cada mini-batch:
$$\hat{\mathbf{z}} = \frac{\mathbf{z} - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} \cdot \boldsymbol{\gamma} + \boldsymbol{\beta}$$

- Reduce el cambio de covariable interno, permite tasas de aprendizaje más altas
- *NO funciona bien con tamaño de batch=1* → usar Layer Norm para batches pequeños o RNN

In [ ]:
# Demostrar el efecto de BatchNorm en el flujo de gradiente
def crear_red_profunda(depth=15, use_bn=False, use_dropout=False, activation='tanh'):
    layers = []
    for i in range(depth):
        layers.append(nn.Linear(64, 64))
        if use_bn: layers.append(nn.BatchNorm1d(64))
        layers.append(nn.Tanh() if activation == 'tanh' else nn.ReLU())
        if use_dropout: layers.append(nn.Dropout(0.3))
    layers.append(nn.Linear(64, 1))
    return nn.Sequential(*layers)

def medir_normas_gradiente(model, X_dummy, y_dummy):
    model.train()
    out  = model(X_dummy)
    loss = F.binary_cross_entropy_with_logits(out.squeeze(), y_dummy)
    loss.backward()
    normas = []
    for m in model.modules():
        if isinstance(m, nn.Linear) and m.weight.grad is not None:
            normas.append(m.weight.grad.norm().item())
    return normas

X_d  = torch.randn(32, 64)
y_d  = torch.randint(0, 2, (32,)).float()

configs = [
    ('Tanh, sin BN',   crear_red_profunda(use_bn=False, activation='tanh')),
    ('Tanh + BN',      crear_red_profunda(use_bn=True,  activation='tanh')),
    ('ReLU, sin BN',   crear_red_profunda(use_bn=False, activation='relu')),
    ('ReLU + BN',      crear_red_profunda(use_bn=True,  activation='relu')),
]

fig, ax = plt.subplots(figsize=(10, 5))
colores = ['steelblue','tomato','seagreen','darkorange']
for (etiqueta, net), color in zip(configs, colores):
    net.zero_grad()
    try:
        gnorms = medir_normas_gradiente(net, X_d, y_d)
        ax.semilogy(gnorms, lw=2, color=color, label=etiqueta)
    except Exception as e:
        print(f'{etiqueta}: {e}')

ax.set(xlabel='Índice de capa (superficial → profunda)',
       ylabel='Norma del gradiente (log)',
       title='Flujo de gradiente en una red de 15 capas\nBN previene el gradiente que se desvanece')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Parte 2 — Aumentación de datos para señales fisiológicas

In [ ]:
def latido_ecg_sintetico(length=250, noise=0.04):
    t = np.arange(length) / 250
    p  = 0.15*np.exp(-((t-0.12)**2)/(2*0.015**2))
    r  = 1.00*np.exp(-((t-0.22)**2)/(2*0.006**2))
    s  = -0.12*np.exp(-((t-0.24)**2)/(2*0.008**2))
    tw = 0.25*np.exp(-((t-0.38)**2)/(2*0.025**2))
    return p + r + s + tw + rng.normal(0, noise, length)

# Definir funciones de aumentación para señales 1-D
aumentaciones = {
    'Original':                  lambda x: x,
    'Escalado de amplitud':      lambda x: x * rng.uniform(0.8, 1.2),
    'Desplaz. temporal (+15)':   lambda x: np.roll(x, 15),
    'Ruido gaussiano':           lambda x: x + rng.normal(0, 0.08, len(x)),
    'Deriva de línea base':      lambda x: x + 0.3*np.sin(2*np.pi*0.3*np.arange(len(x))/250),
    'Recorte+reescalado aleat.': lambda x: np.interp(np.linspace(0,1,len(x)),
                                                   np.linspace(0,1,200),
                                                   x[25:225]),
    'Cutout (parche en cero)':   lambda x: np.concatenate([x[:90], np.zeros(40), x[130:]]),
    'Enmascaramiento frecuencia':lambda x: np.real(np.fft.ifft(
                                 np.fft.fft(x) * (np.arange(len(x)) < 80))),
}

latido = latido_ecg_sintetico()
n_aug = len(aumentaciones)
fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex=True)

for ax, (nombre, aug_fn) in zip(axes.flat, aumentaciones.items()):
    latido_aug = aug_fn(latido.copy())
    ax.plot(latido,     alpha=0.4, color='gray', lw=1, label='Original')
    ax.plot(latido_aug, color='steelblue', lw=1.5, label='Aumentado')
    ax.set(title=nombre, yticks=[])
    ax.legend(fontsize=7, loc='upper right')

plt.suptitle('Estrategias de aumentación de datos para señales de ECG 1-D\n'
             'Aplicadas dinámicamente en el DataLoader', y=1.01)
plt.tight_layout()
plt.show()

## Parte 3 — Programación de la tasa de aprendizaje

In [ ]:
T_max = 100
eta_min, eta_max = 1e-5, 1e-2
warmup_steps = 10

def recocido_coseno(t, T, eta_min, eta_max):
    return eta_min + 0.5*(eta_max - eta_min)*(1 + np.cos(np.pi * t / T))

def calentamiento_coseno(t, warmup, T, eta_min, eta_max):
    if t < warmup:
        return eta_max * t / warmup
    return recocido_coseno(t - warmup, T - warmup, eta_min, eta_max)

def calentamiento_coseno_reinicio(t, warmup, T_0, eta_min, eta_max, T_mult=2):
    t2 = t
    Ti = T_0
    while t2 >= Ti:
        t2 -= Ti
        Ti = int(Ti * T_mult)
    return calentamiento_coseno(t2, warmup, Ti, eta_min, eta_max)

steps = np.arange(T_max)

programaciones = {
    'Constante':                  [eta_max] * T_max,
    'Decaimiento escalonado (×0.5/20ep)': [eta_max * (0.5 ** (s // 20)) for s in steps],
    'Recocido coseno':            [recocido_coseno(s, T_max, eta_min, eta_max) for s in steps],
    'Calentamiento + coseno':     [calentamiento_coseno(s, warmup_steps, T_max, eta_min, eta_max) for s in steps],
    'SGDR (reinicios cálidos)':   [calentamiento_coseno_reinicio(s, 5, 20, eta_min, eta_max) for s in steps],
}

fig, ax = plt.subplots(figsize=(11, 4))
for nombre, sched in programaciones.items():
    ax.plot(steps, sched, lw=2, label=nombre)
ax.set(xlabel='Época', ylabel='Tasa de aprendizaje', yscale='log',
       title='Programaciones de tasa de aprendizaje comunes en DL biomédico')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print('Recomendación: calentamiento + recocido coseno es robusto en la mayoría de tareas.')
print('SGDR con reinicios ayuda a escapar de mínimos locales en paisajes de pérdida complejos.')

## Parte 4 — Grad-CAM desde cero

**Grad-CAM** genera un mapa de calor espacial de qué partes de la entrada influyeron más
en una predicción de clase específica:

$$\alpha_k^c = \frac{1}{Z} \sum_{i,j} \frac{\partial y^c}{\partial A_{ij}^k} \qquad L^c_{\text{Grad-CAM}} = \text{ReLU}\!\left(\sum_k \alpha_k^c A^k\right)$$

donde $A^k$ son los mapas de características de la capa convolucional objetivo y $y^c$
es la puntuación de clase.

In [ ]:
class GradCAM_1D:
    """
    Grad-CAM para CNN 1-D. Se conecta (hooks) a una capa convolucional objetivo
    y calcula mapas de importancia de activación.
    """
    def __init__(self, model, target_layer):
        self.model        = model
        self.target_layer = target_layer
        self.gradients     = None
        self.activations   = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate(self, x, class_idx=None):
        """
        x          : (1, C, L) — una sola muestra
        class_idx  : clase objetivo; si es None, usa argmax
        Retorna    : cam (L,) — mapa de importancia normalizado
        """
        self.model.eval()
        x.requires_grad_(True)
        logits = self.model(x)

        if class_idx is None:
            class_idx = logits.argmax(dim=1).item()

        self.model.zero_grad()
        logits[0, class_idx].backward(retain_graph=True)

        # Promedio global de gradientes sobre el tiempo → pesos por canal
        weights = self.gradients.mean(dim=-1, keepdim=True)  # (1, C, 1)
        cam     = (weights * self.activations).sum(dim=1)    # (1, L_feat)
        cam     = F.relu(cam).squeeze().cpu().numpy()

        # Normalizar a [0, 1]
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam, class_idx


# Cargar (o reentrenar rápidamente) la ECG-CNN de la Sesión 13
class ECG_CNN_15(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv1d(1, 32, 11, padding=5), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2))
        self.conv2 = nn.Sequential(
            nn.Conv1d(32, 64, 7, padding=3), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2))
        self.conv3 = nn.Sequential(
            nn.Conv1d(64, 128, 5, padding=2), nn.BatchNorm1d(128), nn.ReLU(),
            nn.AdaptiveAvgPool1d(8))
        self.head  = nn.Sequential(
            nn.Flatten(), nn.Linear(128*8, 128), nn.ReLU(),
            nn.Dropout(0.4), nn.Linear(128, n_classes))

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        return self.head(x)


# ── Construir y entrenar rápidamente ──
def generar_clase_latido(tipo_latido, n, rng_):
    seg_len = 250
    latidos = []
    for _ in range(n):
        b = latido_ecg_sintetico()
        if tipo_latido == 'V':
            t2 = np.arange(seg_len)/250
            b  = 0.9*np.exp(-((t2-0.22)**2)/(2*0.018**2)) + \
                 -0.5*np.exp(-((t2-0.28)**2)/(2*0.015**2)) + \
                 0.2*np.exp(-((t2-0.40)**2)/(2*0.030**2)) + \
                 rng_.normal(0, 0.05, seg_len)
        latidos.append(b)
    return np.array(latidos)

n_por_clase = {'N': 500, 'S': 80, 'V': 100, 'F': 30, 'Q': 25}
clases = list(n_por_clase.keys())
X_s = np.vstack([generar_clase_latido(c, n, rng) for c, n in n_por_clase.items()])
y_s = np.hstack([np.full(n, k) for k, n in enumerate(n_por_clase.values())]).astype(np.int64)
X_s = (X_s - X_s.mean(1, keepdims=True)) / (X_s.std(1, keepdims=True)+1e-8)
X_s = X_s[:, np.newaxis, :].astype(np.float32)

Xtr, Xte, ytr, yte = train_test_split(X_s, y_s, test_size=0.25, stratify=y_s, random_state=0)
Xtr_t = torch.tensor(Xtr); ytr_t = torch.tensor(ytr)
Xte_t = torch.tensor(Xte)
loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=64, shuffle=True)

ecg_cnn = ECG_CNN_15().to(device)
opt_g   = optim.Adam(ecg_cnn.parameters(), lr=1e-3, weight_decay=1e-4)
wts     = torch.tensor(1.0 / np.array(list(n_por_clase.values())), dtype=torch.float32).to(device)
crit_g  = nn.CrossEntropyLoss(weight=wts / wts.sum())

for ep in range(25):
    ecg_cnn.train()
    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device)
        opt_g.zero_grad()
        crit_g(ecg_cnn(Xb), yb).backward()
        nn.utils.clip_grad_norm_(ecg_cnn.parameters(), 1.0)
        opt_g.step()

print('Entrenamiento completo.')
ecg_cnn.eval()
with torch.no_grad():
    acc = (ecg_cnn(Xte_t.to(device)).argmax(1).cpu().numpy() == yte).mean()
print(f'Exactitud en test: {acc:.3f}')

In [ ]:
# Aplicar Grad-CAM al último bloque convolucional
grad_cam = GradCAM_1D(ecg_cnn, target_layer=ecg_cnn.conv3[0])   # último Conv1d

fig, axes = plt.subplots(len(clases), 1, figsize=(14, 12))

for cls_idx, (cls_name, ax) in enumerate(zip(clases, axes)):
    # Elegir un ejemplo correctamente clasificado
    candidatos  = np.where(yte == cls_idx)[0]
    sample_idx  = candidatos[0]

    x_sample   = torch.tensor(Xte[sample_idx:sample_idx+1], dtype=torch.float32).to(device)
    cam, pred  = grad_cam.generate(x_sample, class_idx=cls_idx)

    beat_np    = Xte[sample_idx, 0]
    t_axis     = np.linspace(0, 1, 250)

    # Sobremuestrear el CAM a la longitud de la señal
    cam_up     = np.interp(np.linspace(0, 1, 250), np.linspace(0, 1, len(cam)), cam)

    # Graficar la señal coloreada por intensidad del CAM
    ax.plot(t_axis, beat_np, 'navy', lw=1.2, alpha=0.6, zorder=1)

    # Superponer mapa de calor como gradiente de color relleno
    for i in range(len(t_axis)-1):
        heat = cam_up[i]
        rgba = cm.Reds(heat)
        ax.fill_between(t_axis[i:i+2],
                         beat_np[i:i+2] - 0.08,
                         beat_np[i:i+2] + 0.08,
                         color=rgba, alpha=0.8, linewidth=0)

    ax.set(ylabel='Amplitud', title=f'Grad-CAM  |  Clase: {cls_name} (predicha: {clases[pred]})')
    if cls_idx == len(clases) - 1:
        ax.set_xlabel('Tiempo (s)')

plt.suptitle('Grad-CAM en CNN de ECG — regiones rojas = más importantes para la clasificación\n'
             '(complejo QRS resaltado para N; QRS más ancho para V, etc.)', y=1.01)
plt.tight_layout()
plt.show()

## Parte 5 — Verificaciones de cordura (sanity checks) para mapas de saliencia

Adebayo et al. (2018) demostraron que muchos métodos de saliencia producen mapas
visualmente plausibles **incluso para redes inicializadas aleatoriamente**. Siempre
ejecuta estas verificaciones antes de confiar en una explicación.

In [ ]:
# Verificación de cordura: ¿cambia Grad-CAM cuando aleatorizamos los pesos del modelo?
ecg_random = ECG_CNN_15().to(device)   # inicializada al azar — sin entrenar
grad_cam_random = GradCAM_1D(ecg_random, target_layer=ecg_random.conv3[0])

sample_idx_n = np.where(yte == 0)[0][0]
x_n = torch.tensor(Xte[sample_idx_n:sample_idx_n+1], dtype=torch.float32).to(device)

cam_entrenado, _ = grad_cam.generate(x_n, class_idx=0)
cam_aleatorio, _  = grad_cam_random.generate(x_n, class_idx=0)

cam_entrenado_up = np.interp(np.linspace(0,1,250), np.linspace(0,1,len(cam_entrenado)), cam_entrenado)
cam_aleatorio_up  = np.interp(np.linspace(0,1,250), np.linspace(0,1,len(cam_aleatorio)),  cam_aleatorio)

beat_n = Xte[sample_idx_n, 0]
t_ax   = np.linspace(0, 1, 250)

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
for ax, cam_u, titulo in zip(axes,
                             [cam_entrenado_up, cam_aleatorio_up],
                             ['Red entrenada (CAM debería resaltar el QRS)',
                              'Red inicializada al azar (CAM debería verse distinto)']):
    ax.plot(t_ax, beat_n, 'navy', lw=1.5, alpha=0.6)
    ax2 = ax.twinx()
    ax2.fill_between(t_ax, cam_u, alpha=0.4, color='tomato', label='CAM')
    ax2.set_ylabel('Valor del CAM')
    ax.set(ylabel='Amplitud (norm.)', title=titulo)

plt.suptitle('Verificación de cordura: Grad-CAM debería diferir entre red entrenada y aleatoria\n'
             'Si no → el método de explicación no es significativo', y=1.01)
plt.tight_layout()
plt.show()

correlacion = np.corrcoef(cam_entrenado_up, cam_aleatorio_up)[0,1]
print(f'Correlación entre Grad-CAM entrenado y aleatorio: {correlacion:.3f}')
print('Correlación baja → Grad-CAM es sensible a los pesos aprendidos ✅')
print('Correlación alta → la explicación puede no reflejar la lógica del modelo ❌')

## ✏️ Ejercicios

1. **Ubicación del dropout.** Compara colocar dropout (a) después de cada capa,
   (b) solo antes de la FC final, y (c) usando DropBlock (dropout espacial para CNN)
   en el clasificador de ECG. Reporta el AUROC en test y la forma de la curva de
   entrenamiento para cada uno.

2. **Aumentación Mixup.** Implementa Mixup: $\tilde{x} = \lambda x_i + (1-\lambda) x_j$,
   $\tilde{y} = \lambda y_i + (1-\lambda) y_j$. Aplícalo al entrenamiento de latidos
   de ECG y compáralo con la aumentación estándar. ¿Ayuda el entrenamiento con
   etiquetas blandas con las anotaciones AAMI ruidosas?

3. **Guided Grad-CAM.** Implementa Guided Backpropagation (poner a cero los gradientes
   negativos en las ReLU durante el paso hacia atrás). Combínalo con Grad-CAM mediante
   multiplicación elemento a elemento (Guided Grad-CAM). Compara los mapas de saliencia
   resultantes en nitidez y plausibilidad biológica.

4. **Layer Norm vs Batch Norm.** La BiLSTM de la Sesión 14 no usa normalización.
   Añade (a) Batch Norm aplicada después de las salidas de la LSTM, (b) Layer Norm.
   Compara la velocidad de convergencia y el AUROC. Explica teóricamente por qué
   Layer Norm se prefiere sobre Batch Norm para modelos recurrentes.

5. *(Desafío)* **Mapas de activación de clase para fMRI.** Descarga un dataset simple de
   fMRI (ej., Haxby 2001 — disponible vía nilearn). Entrena una CNN 3-D (usa un ResNet
   2-D preentrenado en cortes axiales como línea base). Aplica Grad-CAM y visualiza el
   volumen de activación superpuesto en un cerebro template MNI usando las funciones de
   graficado de nilearn. Interpreta qué regiones impulsan la clasificación.

## 📚 Conjuntos de datos

| Conjunto de datos | Fuente | Notas |
|---|---|
| MIT-BIH Arrhythmia | https://physionet.org/content/mitdb/ | Clasificación de arritmias |
| Haxby fMRI (nilearn) | `nilearn.datasets.fetch_haxby()` | Reconocimiento de objetos fMRI, clásico |
| ISIC skin lesion | https://challenge.isic-archive.com | Validación de Grad-CAM en dermatoscopia |